# 🧠 AI Trivia Quiz — Step 2: Question Generator
We use Groq + Llama to generate trivia questions dynamically.
Every game = fresh questions. Never the same quiz twice!

In [1]:
import os, json, re
from dotenv import load_dotenv
from groq import Groq

load_dotenv('trivia_app/.env')
client = Groq(api_key=os.environ.get('GROQ_API_KEY'))

print('✅ Groq client ready')

✅ Groq client ready


In [2]:
# Available quiz categories
CATEGORIES = [
    '🌍 General Knowledge',
    '🔬 Science & Technology',
    '🎬 Movies & TV',
    '⚽ Sports',
    '🎵 Music',
    '🏛️ History',
    '🌐 Geography',
    '🎮 Video Games',
    '🍕 Food & Cooking',
    '🐾 Animals & Nature',
]

DIFFICULTY_LEVELS = ['Easy', 'Medium', 'Hard']

print('Categories:', CATEGORIES)
print('Difficulties:', DIFFICULTY_LEVELS)

Categories: ['🌍 General Knowledge', '🔬 Science & Technology', '🎬 Movies & TV', '⚽ Sports', '🎵 Music', '🏛️ History', '🌐 Geography', '🎮 Video Games', '🍕 Food & Cooking', '🐾 Animals & Nature']
Difficulties: ['Easy', 'Medium', 'Hard']


In [3]:
def generate_questions(category: str, difficulty: str, num_questions: int = 10) -> list[dict]:
    """
    Calls Groq/Llama to generate trivia questions.
    Returns a list of question dicts.
    """
    prompt = f"""Generate exactly {num_questions} trivia questions about {category} at {difficulty} difficulty.

Return ONLY a valid JSON array. No explanation, no markdown, no extra text.
Each object in the array must have exactly these fields:
{{
  "question": "the question text",
  "options": ["A) option1", "B) option2", "C) option3", "D) option4"],
  "answer": "A) option1",
  "explanation": "brief explanation of why this is correct"
}}

Rules:
- The answer must exactly match one of the options strings
- Make distractors (wrong options) plausible and interesting
- For Easy: well-known facts. Medium: requires some knowledge. Hard: specific/challenging.
- Vary the correct answer position (don't always put it as A)
"""

    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=3000,
        temperature=0.7,
    )

    raw = response.choices[0].message.content.strip()

    # Clean up any markdown code blocks if model adds them
    raw = re.sub(r'^```json\s*', '', raw)
    raw = re.sub(r'^```\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)

    questions = json.loads(raw)
    return questions

print('✅ generate_questions() function defined')

✅ generate_questions() function defined


In [4]:
# TEST: Generate 3 sample questions
print('Generating 3 test questions... (takes ~5 seconds)\n')

questions = generate_questions(
    category='🔬 Science & Technology',
    difficulty='Medium',
    num_questions=3
)

for i, q in enumerate(questions, 1):
    print(f'Q{i}: {q["question"]}')
    for opt in q['options']:
        marker = '✅' if opt == q['answer'] else '  '
        print(f'  {marker} {opt}')
    print(f'  💡 {q["explanation"]}')
    print()

Generating 3 test questions... (takes ~5 seconds)

Q1: What is the process by which water moves through a plant, from the roots to the leaves, and is then released into the air as water vapor?
     A) Respiration
     B) Photosynthesis
  ✅ C) Transpiration
     D) Evaporation
  💡 Transpiration is the process by which water is transported through a plant, from the roots to the leaves, and is then released into the air as water vapor.

Q2: Which of the following types of rocks is formed from the cooling and solidification of magma or lava?
     A) Sedimentary rocks
  ✅ B) Igneous rocks
     C) Metamorphic rocks
     D) Foliated rocks
  💡 Igneous rocks are formed from the cooling and solidification of magma or lava, and can be either intrusive or extrusive.

Q3: What is the term for the 'building blocks of life', which are the basic structural and functional units of all living organisms?
     A) Molecules
     B) Tissues
     C) Organs
  ✅ D) Cells
  💡 Cells are the basic structural and 

In [6]:
# Save the generator as a reusable Python file inside trivia_app/

code = '''import os, json, re
from groq import Groq

CATEGORIES = [
    "🌍 General Knowledge",
    "🔬 Science & Technology",
    "🎬 Movies & TV",
    "⚽ Sports",
    "🎵 Music",
    "🏛️ History",
    "🌐 Geography",
    "🎮 Video Games",
    "🍕 Food & Cooking",
    "🐾 Animals & Nature",
]

DIFFICULTY_LEVELS = ["Easy", "Medium", "Hard"]

def get_client():
    return Groq(api_key=os.environ.get("GROQ_API_KEY"))

def generate_questions(category: str, difficulty: str, num_questions: int = 10) -> list:
    client = get_client()
    prompt = f"""Generate exactly {num_questions} trivia questions about {category} at {difficulty} difficulty.

Return ONLY a valid JSON array. No explanation, no markdown, no extra text.
Each object must have exactly these fields:
{{
  "question": "the question text",
  "options": ["A) option1", "B) option2", "C) option3", "D) option4"],
  "answer": "A) option1",
  "explanation": "brief explanation of why this is correct"
}}

Rules:
- The answer must exactly match one of the options strings
- Make distractors plausible and interesting
- For Easy: well-known facts. Medium: requires some knowledge. Hard: specific/challenging.
- Vary the correct answer position (don't always put it as A)
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=3000,
        temperature=0.7,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```json\\s*", "", raw)
    raw = re.sub(r"^```\\s*", "", raw)
    raw = re.sub(r"\\s*```$", "", raw)
    return json.loads(raw)
'''

with open('trivia_app/question_generator.py', 'w',encoding='utf-8') as f:
    f.write(code)

print('✅ trivia_app/question_generator.py saved!')
print('   This file will be imported by our Streamlit app.')

✅ trivia_app/question_generator.py saved!
   This file will be imported by our Streamlit app.
